# 🚀 Huấn luyện Model Pothole YOLOv11n (Google Colab T4 GPU)
Notebook này được tối ưu tự động phát hiện đường dẫn dataset (không bao giờ lỗi `FileNotFoundError`).

In [ ]:
# Bước 1: Cài đặt thư viện ultralytics
!pip install ultralytics

In [ ]:
# Bước 2: Kiểm tra GPU T4
import torch
print(f"CUDA: {torch.cuda.is_available()} - {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CHƯA BẬT GPU'}")

In [ ]:
# Bước 3: Tự động tìm file zip, giải nén và xác định đường dẫn dataset chuẩn xác 100%
import os, glob, zipfile, sys

# 1. Tìm tất cả các file zip trong thư mục làm việc hiện tại
zip_files = glob.glob('*.zip') + glob.glob('/content/*.zip')
print(f"Các file zip tìm thấy: {zip_files}")

target_dataset_dir = '/content/dataset' if os.path.exists('/content') else './dataset'

# 2. Giải nén nếu có file zip
if zip_files:
    chosen_zip = zip_files[0]
    print(f"Đang giải nén {chosen_zip} vào {target_dataset_dir}...")
    with zipfile.ZipFile(chosen_zip, 'r') as z:
        z.extractall(target_dataset_dir)
    print("Giải nén thành công!")

# 3. Quét tìm vị trí thực tế của thư mục chứa images/train
found_root = None
search_base = '/content' if os.path.exists('/content') else '.'
for root, dirs, files in os.walk(search_base):
    if 'images' in dirs and 'labels' in dirs:
        # Kiểm tra xem có train và valid không
        parent = os.path.abspath(os.path.join(root, '..'))
        if os.path.exists(os.path.join(parent, 'train', 'images')):
            found_root = parent
            break
        elif os.path.exists(os.path.join(root, 'train', 'images')):
            found_root = root
            break

if not found_root:
    # Fallback kiểm tra target_dataset_dir
    if os.path.exists(os.path.join(target_dataset_dir, 'train', 'images')):
        found_root = os.path.abspath(target_dataset_dir)
    elif os.path.exists('/content/train/images'):
        found_root = '/content'

if not found_root:
    print("❌ LỖI: Chưa thấy thư mục train/images! Bạn đã upload file zip lên Colab chưa?")
else:
    print(f"✅ Đã định vị chính xác dataset tại: {found_root}")

    # Tạo pothole_data.yaml với đường dẫn tuyệt đối chuẩn xác
    train_p = os.path.join(found_root, 'train', 'images').replace('\\', '/')
    val_p = os.path.join(found_root, 'valid', 'images').replace('\\', '/')
    test_p = os.path.join(found_root, 'test', 'images').replace('\\', '/')

    # Nếu không có valid, dùng test thay thế
    if not os.path.exists(val_p.replace('/', os.sep)):
        print("⚠️ Không thấy thư mục valid, tự động chuyển sang dùng test làm validation set.")
        val_p = test_p

    yaml_content = f"""# Auto-generated YAML
train: {train_p}
val: {val_p}
test: {test_p}

nc: 1
names: ['pothole']
"""
    with open('pothole_data.yaml', 'w') as f:
        f.write(yaml_content.strip())
    print("✅ Đã ghi file pothole_data.yaml:")
    print(yaml_content)

In [ ]:
# Bước 4: Tiến hành Huấn Luyện (50 Epochs)
from ultralytics import YOLO

model = YOLO('yolo11n.pt')

results = model.train(
    data='pothole_data.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    optimizer='AdamW',
    lr0=0.001,
    project='runs/pothole',
    name='pothole_v1',
    save=True,
    plots=True,
    patience=15
)

In [ ]:
# Bước 5: Tải file pothole_best.pt về máy
import shutil, glob
from google.colab import files

# Tìm file best.pt vừa train xong
weights = glob.glob('runs/**/weights/best.pt', recursive=True)
if weights:
    best_weight = weights[-1]
    shutil.copy(best_weight, 'pothole_best.pt')
    print(f"✅ Đã copy ra pothole_best.pt từ {best_weight}")
    files.download('pothole_best.pt')
else:
    print("❌ Không tìm thấy file weights/best.pt")